# Last Mile Delivery Report Automation by 3PL
**Objective:** Automate the data preprocessing of daily last mile delivery to generate formatted Excel files based on 3PL vendor names, delivery dates, and time slots.

**Core ETL Components:**
1. **Extract:** Ingest raw CSV daily delivery data via Google Colab upload.
2. **Transform:** Clean data types, filter specific columns, sort logically, and apply conditional business formatting (B2B & COD orders).
3. **Load:** Distribute the processed data into vendor-specific Excel files, format columns, and compress into a single ZIP archive for distribution.

# Setup, Imports, and Configuration

In [ ]:
import os
import shutil
from zipfile import ZipFile
from datetime import datetime
import pandas as pd
import numpy as np
import pytz
import openpyxl
from google.colab import files

# --- CONFIGURATION ---
LOCAL_TZ = pytz.timezone('Asia/Jakarta')
DATE_FORMAT = "%d-%m-%Y"
CURRENT_DATE_STR = datetime.now(LOCAL_TZ).strftime(DATE_FORMAT)

UPLOAD_DIR = '/content/'
EXPORT_DIR = '/content/exported_files'

# Desired columns for the final output
TARGET_COLUMNS = [
    'driver_name', 'district', 'customer_name', 'delivery_date',
    'address', 'order_no', 'packaging_option','distance_in_km', 
    'hubs', 'total_price', 'time_slot',
    'payment_method', 'shipping_number (Box #)'
]

# Ensure export directory exists
os.makedirs(EXPORT_DIR, exist_ok=True)
print(f"Setup complete. Current processing date: {CURRENT_DATE_STR}")

# 1. Extract
Upload the raw CSV file into the Colab environment and load it into a Pandas DataFrame.

In [ ]:
def extract_data(upload_dir: str) -> pd.DataFrame:
    print("Please upload the daily delivery CSV file:")
    uploaded = files.upload()
    
    csv_files = [f for f in os.listdir(upload_dir) if f.endswith('.csv')]
    if not csv_files:
        raise FileNotFoundError("No CSV files found in the upload directory.")
        
    file_path = os.path.join(upload_dir, csv_files[0])
    df = pd.read_csv(file_path)
    print(f"Successfully extracted {len(df)} rows from {csv_files[0]}")
    return df

# Execute Extract
raw_df = extract_data(UPLOAD_DIR)

# 2. Transform
Clean data types, filter the necessary columns, and sort the data for operational efficiency.

In [ ]:
def clean_and_transform(df: pd.DataFrame, columns: list) -> pd.DataFrame:
    """Applies type casting, column filtering, and sorting."""
    # Always operate on a copy to preserve raw data
    transformed_df = df.copy()

    # 1. Transform columns
    # Transform values in 'total_price' column by filling nan values with 0 and convert data type into integer
    transformed_df.loc[:, 'total_price'] = (
        transformed_df['total_price']
        .fillna(0)
        .astype(str)
        .str.replace(',', '', regex=False)
        .astype(float)
        .astype(int)
    )
    # Convert 'driver_name' to uppercase and remove 'MITRA-' or 'mitra-' prefix
    transformed_df.loc[:, 'driver_name'] = (
        transformed_df['driver_name']
        .str.upper()
        .str.replace('^(MITRA-|mitra-)', '', regex=True)
        )
    # Replace 'ALL' in 'packaging_option' with 'BIGBOX + BIGBOX'
    transformed_df.loc[:, 'packaging_option'] = (
        transformed_df['packaging_option']
        .str.replace('^(ALL)', 'BIGBOX + BIGBOX', regex=True)
        )
    # Round 'distance_in_km' to 2 decimal places
    transformed_df.loc[:, 'distance_in_km'] = (
        transformed_df['distance_in_km'].round(2)
        )
    # Replace specific time slot variations with standardized values
    transformed_df.loc[:, 'time_slot'] = (
        transformed_df['time_slot']
        .str.replace('^(slot-12bb)', 'slot-1bb', regex=True)
        .str.replace('^(slot-b2b-2)', 'slot-0bb', regex=True)
        )
    
    # 3. Filter Columns
    valid_columns = [col for col in columns if col in transformed_df.columns]
    transformed_df = transformed_df[valid_columns]

    # 4. Sort Data logically for logistics
    if all(col in transformed_df.columns for col in ['hubs', 'driver_name', 'customer_name']):
        transformed_df = transformed_df.sort_values(
            by=['hubs', 'driver_name', 'customer_name'], 
            ascending=[True, True, True]
        )

    return transformed_df

# Execute Transform
clean_df = clean_and_transform(raw_df, TARGET_COLUMNS)
display(clean_df.head(3))

# 3. Formatting & Load Phase (Vendor Distribution)
Apply business-logic styling (B2B & COD) and split the master dataframe into vendor-specific Excel files.

In [ ]:
def highlight_b2b_and_payment(row: pd.Series) -> list[str]:
    """Conditional formatting based on time_slot and payment_method."""
    time_slot_value = row.get('time_slot', '')
    payment_method_value = row.get('payment_method', '')

    background_color = '#FFFFFF'  
    text_color = '#000000'        

    if time_slot_value in ('slot-0bb', 'slot-1bb'):
        background_color = '#7393B3' 
    elif payment_method_value == 'Cash on Delivery':
        text_color = 'green'

    return [f'background-color: {background_color}; color: {text_color}; border: 0.5px solid black' for _ in row]

In [ ]:
# Call the function for each vendor and time slot
# 3PL A
export_vendor_list_delivery(clean_df, '3PLA', '3PLA', 'slot-0|slot-0bb', 'DINI HARI')
export_vendor_list_delivery(clean_df, '3PLA', '3PLA', 'slot-1$|slot-1bb', 'PAGI')
export_vendor_list_delivery(clean_df, '3PLA', '3PLA', 'slot-2|slot-sameday03', 'SORE')
export_vendor_list_delivery(clean_df, '3PLA', '3PLA', 'slot-13|slot-sameday$', 'MALAM')

# 3PL B
export_vendor_list_delivery(clean_df, '3PLB', '3PLB', 'slot-0|slot-0bb', 'DINI HARI')
export_vendor_list_delivery(clean_df, '3PLB', '3PLB', 'slot-1$|slot-1bb', 'PAGI')
export_vendor_list_delivery(clean_df, '3PLB', '3PLB', 'slot-2|slot-sameday03', 'SORE')
export_vendor_list_delivery(clean_df, '3PLB', '3PLB', 'slot-13|slot-sameday$', 'MALAM')

# 3PL C
export_vendor_list_delivery(clean_df, '3PLC', '3PLC', 'slot-0|slot-0bb', 'DINI HARI')
export_vendor_list_delivery(clean_df, '3PLC', '3PLC', 'slot-1$|slot-1bb', 'PAGI')
export_vendor_list_delivery(clean_df, '3PLC', '3PLC', 'slot-2|slot-sameday03', 'SORE')
export_vendor_list_delivery(clean_df, '3PLC', '3PLC', 'slot-13|slot-sameday$', 'MALAM')

# 4. Post-Processing & Archiving
Handle the specialized Cash on Delivery (COD) report and zip all daily files for download.

In [ ]:
def generate_cod_report(df: pd.DataFrame):
    """Filters COD orders, exports them, and uses openpyxl to auto-fit columns."""
    cod_cols = ['driver_name', 'customer_name', 'delivery_date', 'hubs', 'order_no', 
                'total_price', 'time_slot', 'payment_method', 'shipping_number (Box #)']
    
    # Filter for COD and specific slots
    cod_df = df[
        (df['payment_method'] == 'Cash on Delivery') &
        (~df['time_slot'].str.contains('b'))
    ]
    
    valid_cols = [col for col in cod_cols if col in cod_df.columns]
    cod_df = cod_df[valid_cols]
    
    if cod_df.empty:
        print("No COD orders found matching criteria.")
        return

    filepath = os.path.join(f'EXPORT_DIR, "Cash on Delivery Orders{CURRENT_DATE_STR}.xlsx"')
    
    def cod_style(row):
        return [f"color: green; border: 0.5px solid black" for _ in row]
        
    cod_df.style.apply(cod_style, axis=1).to_excel(filepath, index=False)
    
    # Post-process with openpyxl for alignment and width
    wb = openpyxl.load_workbook(filepath)
    ws = wb.active
    
    for row in ws.iter_rows():
        for cell in row:
            cell.alignment = openpyxl.styles.Alignment(horizontal='center', vertical='center')
            
    for column in ws.columns:
        max_length = max((len(str(cell.value)) for cell in column if cell.value), default=0)
        ws.column_dimensions[column[0].column_letter].width = max_length + 1.5
        
    wb.save(filepath)
    print("COD Report generated and formatted.")

def package_and_download(export_dir: str, zip_name: str):
    """Zips the export directory and triggers browser download."""
    with ZipFile(zip_name, 'w') as zipf:
        for root, _, files_list in os.walk(export_dir):
            for file in files_list:
                file_path = os.path.join(root, file)
                zipf.write(file_path, os.path.relpath(file_path, export_dir))
                
    print(f"Packaged {len(os.listdir(export_dir))} files into {zip_name}")
    files.download(zip_name)

# Execute COD Report and Zip Archive
generate_cod_report(clean_df)
package_and_download(EXPORT_DIR, f"3pl_list_delivery_{CURRENT_DATE_STR}.zip")